Regressor Model Build and Testing:
* LightGBM Regressor
* Random Forest Regressor

Testing using regression models of gradient boosting and random forest on CVD risk score, as opposed to CVD risk level.

In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report, r2_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor

In [18]:
# use cvd risk level not cvd risk score
# blood pressure categories found from https://www.heart.org/en/health-topics/high-blood-pressure/understanding-blood-pressure-readings
# remove systolic, diastolic, and blood pressure, just keep blood pressure category (determined by systolic over diastolic values)

df = pd.read_csv('cvd_dataset.csv')
df_edited = df.drop(columns=['Blood Pressure (mmHg)','Systolic BP','Diastolic BP'])
df_edited = df_edited.dropna(subset=['CVD Risk Score'])

# ordinal encoding categories
sex_categories = ['F','M'] 
physical_activity_categories = ['Low','Moderate','High']
blood_pressure_categories = ['Normal','Elevated','Hypertension Stage 1','Hypertension Stage 2']
# smoking, diabetes, family history all Y/N (0,1)
yn_categories = ['Y','N']

encoder = OrdinalEncoder(categories=[sex_categories, physical_activity_categories, blood_pressure_categories, yn_categories, yn_categories, yn_categories])
df_edited[['Sex','Physical Activity Level','Blood Pressure Category','Smoking Status','Diabetes Status','Family History of CVD']] = encoder.fit_transform(df_edited[['Sex','Physical Activity Level','Blood Pressure Category','Smoking Status','Diabetes Status','Family History of CVD']])

X = df_edited.drop(columns=['CVD Risk Level','CVD Risk Score'])
y = df_edited['CVD Risk Score']

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

all_y_true = []
xgb_y_pred = []
xgb_r2 = []
rf_r2 = []

fold_num = 1
for train_index, test_index in kf.split(X,y): # splits dataset into stratified train-test indices
    X_train, X_test = X.iloc[train_index], X.iloc[test_index] # features
    y_train, y_test = y.iloc[train_index], y.iloc[test_index] # labels

    xgb_model = xgb.XGBRegressor(objective='reg:squarederror',
                                 n_estimators=200, # higher values may improve performance, risks overfitting
                                 random_state=42,
                                 learning_rate=0.1,
                                 max_depth=1)
    xgb_model.fit(X_train, y_train)

    random_forest_model = RandomForestRegressor(n_estimators=200,
                                                random_state=42,
                                                max_depth=10)
    random_forest_model.fit(X_train, y_train)

    y_pred_xgb = xgb_model.predict(X_test)
    y_pred_random_forest = random_forest_model.predict(X_test)

    r2_xgb = r2_score(y_test, y_pred_xgb)
    r2_random = r2_score(y_test, y_pred_random_forest)

    xgb_r2.append(r2_xgb)
    rf_r2.append(r2_random)

print("Average XGB R2 Score: ", np.mean(xgb_r2))
print("Average Random Forest R2 Score: ", np.mean(rf_r2))



Average XGB R2 Score:  0.8133021070207602
Average Random Forest R2 Score:  0.7857144351747072
